In [198]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
import random
import pickle
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import r2_score
import seaborn as sns
import math
import ast

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split


from data.data_loader_extend import EPCDataset, UMassDataset
from models.utils_extend import create_model, train_for_long_term_forecast, train_for_short_term_forecast, evaluate_for_long_term_forecast, evaluate_for_short_term_forecast


from explainers.utils_extend import get_explainer, unpack_eval_for_single


In [152]:
lc_output_file = "results/umass_important_features_for_lf_all.txt"
sc_output_file = "results/umass_important_features_for_sf_all.txt"
umass_path = 'data/umass/HomeA/HomeA_with_weather.csv'

def get_base_features(path):
    df = pd.read_csv(path)
    features = df.columns.tolist()
    return [f.lower() for f in features]

base_features = get_base_features(umass_path)
time_features = ['sin_hour', 'cos_hour', 'sin_day', 'cos_day', 'sin_month', 'cos_month', 'hour', 'day', 'month']
base_features += time_features

base_features

['use_total',
 'datetime',
 'use_m2',
 'furnacehrv_m2',
 'cellaroutlets_m2',
 'washingmachine_m2',
 'fridgerange_m2',
 'disposaldishwasher_m2',
 'kitchenlights_m2',
 'bedroomoutlets_m2',
 'bedroomlights_m2',
 'masteroutlets_m2',
 'masterlights_m2',
 'ductheaterhrv_m2',
 'use_m3',
 'electricrange_m3',
 'dryer_m3',
 'garagemudroomlights_m3',
 'diningroomoutlets_m3',
 'mudroomoutlets_m3',
 'masterbathoutlets_m3',
 'garageoutlets_m3',
 'basementoutdooroutlets_m3',
 'use_m4',
 'kitchendenlights_m4',
 'masterbedbathlights_m4',
 'masteroutlets_m4',
 'denoutdoorlights_m4',
 'denoutlets_m4',
 'rearbasementlights_m4',
 'kitchenoutletseast_m4',
 'kitchenoutletssouth_m4',
 'dishwasherdisposalsinklight_m4',
 'refrigerator_m4',
 'microwave_m4',
 'officelights_m4',
 'temperature',
 'humidity',
 'visibility',
 'apparenttemperature',
 'pressure',
 'windspeed',
 'cloudcover',
 'windbearing',
 'precipintensity',
 'dewpoint',
 'precipprobability',
 'icon',
 'sin_hour',
 'cos_hour',
 'sin_day',
 'cos_day',

In [153]:
model_path_pattern = re.compile(r"\./trained_models/(.+)")
feature_pattern = re.compile(r"([\w_]+): ([\d\.]+)")

important_features_dict = {}
current_model = None
current_section = None
                    
with open(lc_output_file, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            
            # 모델 경로 감지 및 모델 키 추출
            match = model_path_pattern.match(line)
            if match:
                current_model = match.group(1)
                important_features_dict[current_model] = {"long": {}, "short": {}}
                continue

            # 섹션 감지
            if "Important Long-term Features" in line:
                current_section = "long"
                continue
            elif "Important Short-term Features" in line:
                current_section = "short"
                continue
            
            # Feature 값 추출
            match = feature_pattern.match(line)
            if match and current_model and current_section:
                feature_name, score = match.groups()
                if feature_name in base_features:
                    important_features_dict[current_model][current_section][feature_name] = score


In [154]:
important_features_dict

{'LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth': {'long': {'masteroutlets_m4': '6.560765200933008',
   'officelights_m4': '6.486507020802914',
   'denoutdoorlights_m4': '6.314165264872591',
   'dishwasherdisposalsinklight_m4': '6.248779230395425',
   'refrigerator_m4': '6.190824777963898',
   'kitchendenlights_m4': '6.137539853990391',
   'basementoutdooroutlets_m3': '6.053651632572292',
   'microwave_m4': '6.028595263838842',
   'kitchenoutletseast_m4': '6.028381914245709',
   'denoutlets_m4': '6.02292912442272',
   'masterbathoutlets_m3': '6.022492450936725',
   'mudroomoutlets_m3': '5.969695622219534',
   'use_m4': '5.885407986215786',
   'rearbasementlights_m4': '5.8785412967143',
   'diningroomoutlets_m3': '5.867933058691355',
   'kitchenoutletssouth_m4': '5.847786414713709',
   'masterbedbathlights_m4': '5.656076017994024',
   'garageoutlets_m3': '5.598190966397515',
   'sin_month': '5.403649017972079',
   'cos_month': '4.2919675735112435',
   'use_total': '

In [155]:
### method 1: long, short 각각 중요도 점수 비율 
def select_features_by_termwise_ratio(scores, threshold_ratio=0.7):
    sorted_feats = sorted(scores.items(), key=lambda x: float(x[1]), reverse=True)
    total = sum(float(v) for _, v in sorted_feats)
    selected = {}
    running_sum = 0
    for f, v in sorted_feats:
        running_sum += float(v)
        selected[f] = v
        # selected.append(f)
        if running_sum / total >= threshold_ratio:
            break
    return selected


### method 2: long, short 중요도 합의 점수 비율
def select_features_by_combined_score(long_scores, short_scores, threshold_ratio=0.7):
    all_features = set(long_scores) | set(short_scores)
    combined_scores = {
        f: float(long_scores.get(f, 0)) + float(short_scores.get(f, 0))
        for f in all_features
    }
    sorted_feats = sorted(combined_scores.items(), key=lambda x: float(x[1]), reverse=True)
    total = sum(score for _, score in sorted_feats)
    selected = {}
    running_sum = 0
    for f, score in sorted_feats:
        running_sum += score
        selected[f] = score
        # selected.append(f)
        if running_sum / total >= threshold_ratio:
            break
    return selected


### method 3: long, short 중요도 순위 합
def select_by_rank_sum(long_ranked: dict, short_ranked: dict, max_rank_sum: int = 40):
    long_ranked_key = list(long_ranked.keys())
    short_ranked_key = list(short_ranked.keys())
    
    long_ranked = sorted(long_ranked.items(), key=lambda x: float(x[1]), reverse=True)
    
    selected = {}
    for feat, l_score in long_ranked:
        if feat in short_ranked:
            long_rank = long_ranked_key.index(feat) + 1  # 순위는 1부터
            short_rank = short_ranked_key.index(feat) + 1
            if long_rank + short_rank <= max_rank_sum:
                selected[feat] = l_score
    return selected


### method 4: long, short 중요도 순위 상대 거리
def select_by_relative_rank_distance(
    base_ranked_dict: dict,
    base_ranked: list,  # 중심 기준 (long or short)
    reference_ranked: list,  # 상대 비교 term
    top_k: int = 40,
    max_distance: int = 6
):
    # base_ranked_dict = sorted(base_ranked.items(), key=lambda x: float(x[1]), reverse=True)
    
    selected = {}
    for i, feat in enumerate(base_ranked[:top_k]):  # 기준 term에서 상위 top_k만
        if feat in reference_ranked:
            ref_rank = reference_ranked.index(feat)
            if abs(ref_rank - i) <= max_distance:
                selected[feat] = base_ranked_dict[feat]
    return selected

In [255]:
extracted_features = {}


filtered_features_0 = {}
filtered_features_1 = {}
filtered_features_2 = {}
filtered_features_3 = {}
filtered_features_4 = {}

for model_name, all_features in important_features_dict.items():
    filtered_features_0[model_name] = {}
    filtered_features_0[model_name]['long'] = dict(list(all_features['long'].items())[:7])
    filtered_features_0[model_name]['short'] = dict(list(all_features['short'].items())[:7])

    
    filtered_features_1[model_name] = {}
    long_selected = select_features_by_termwise_ratio(all_features['long'])
    short_selected = select_features_by_termwise_ratio(all_features['short'])
    filtered_features_1[model_name]['long'] = dict(list(long_selected.items())[:7])
    filtered_features_1[model_name]['short'] = dict(list(short_selected.items())[:7])

    ### method 2
    filtered_features_2[model_name] = {}
    selected_feastures = select_features_by_combined_score(all_features['long'], all_features['short'])
    filtered_features_2[model_name]['long'] = dict(list(long_selected.items())[:7])
    filtered_features_2[model_name]['short'] = dict(list(short_selected.items())[:7])

    ### method 3
    filtered_features_3[model_name] = {}
    selected_feastures = select_by_rank_sum(all_features['long'], all_features['short'])
    filtered_features_3[model_name]['long'] = dict(list(selected_feastures.items())[:7])
    filtered_features_3[model_name]['short'] = dict(list(selected_feastures.items())[:7])

    ### method 4
    filtered_features_4[model_name] = {}
    long_selected = select_by_relative_rank_distance(all_features['long'], list(all_features['long']), list(all_features['short']))
    short_selected = select_by_relative_rank_distance(all_features['short'], list(all_features['short']), list(all_features['long']))
    filtered_features_4[model_name]['long'] = dict(list(long_selected.items())[:7])
    filtered_features_4[model_name]['short'] = dict(list(short_selected.items())[:7])

In [256]:
extracted_features['ALL'] = filtered_features_0
extracted_features['STP'] = filtered_features_1
extracted_features['MTP'] = filtered_features_2
extracted_features['RS'] = filtered_features_3
extracted_features['RSF'] = filtered_features_4

In [257]:
extracted_features['STP']

{'LS_CNNLSTM_umass_long2160_short720_pred256_hs256_nl3_a0.3_b1.0.pth': {'long': {'masteroutlets_m4': '6.560765200933008',
   'officelights_m4': '6.486507020802914',
   'denoutdoorlights_m4': '6.314165264872591',
   'dishwasherdisposalsinklight_m4': '6.248779230395425',
   'refrigerator_m4': '6.190824777963898',
   'kitchendenlights_m4': '6.137539853990391',
   'basementoutdooroutlets_m3': '6.053651632572292'},
  'short': {'refrigerator_m4': '0.5599940304941764',
   'use_total': '0.5564946381698007',
   'mudroomoutlets_m3': '0.5462559839142147',
   'diningroomoutlets_m3': '0.5348637312692958',
   'kitchenoutletssouth_m4': '0.5334012683227083',
   'rearbasementlights_m4': '0.5192918468022713',
   'masterbathoutlets_m3': '0.5079581808627823'}},
 'LS_CNNLSTM_umass_long2160_short720_pred256_hs512_nl3_a0.3_b1.0.pth': {'long': {'dishwasherdisposalsinklight_m4': '8.936097393260875',
   'masteroutlets_m4': '8.501297783126324',
   'kitchenoutletssouth_m4': '8.394785508549345',
   'microwave_m4':

In [258]:
def get_gpt_features_with_scores(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    
    gpt_features = ast.literal_eval(content)
    
    for features_by_model in gpt_features.values():
        for model_path, features in features_by_model.items():
            if 'LS' not in model_path: continue
            feat_with_scores = important_features_dict[model_path]
            for term, feat_list in features.items():
                new_feat_dict = {}
                for feat in feat_list:
                    new_feat_dict[feat] = feat_with_scores[term][feat]
                features[term] = new_feat_dict
    
    return gpt_features

In [259]:
gpt_features = get_gpt_features_with_scores("results/fs_umass_by_gpt_for_CIS.txt")

extracted_features = extracted_features | gpt_features

In [260]:
def compute_CIS(count_dict):
    CIS_dict = {}
    
    for feat_name, cnt_score in count_dict.items():
        CIS_dict[feat_name] = 0

        cnt = cnt_score['count']
        score = cnt_score['score']
        
        cis_i = (score / cnt) * math.log(1+cnt)
        
        CIS_dict[feat_name] = cis_i

    return CIS_dict

In [261]:
CIS_by_fs = {}

for fs_name, all_info in extracted_features.items():
    CIS_by_fs[fs_name] = {}
    for model_path, features in all_info.items():
        for term, feats in features.items():
            feat_cnt_and_scoresum = {}
            # CIS_by_fs[fs_name][term] = {}
            for feat_name, score in feats.items():
                if feat_name in feat_cnt_and_scoresum.keys():
                    feat_cnt_and_scoresum[feat_name]['count'] += 1
                    feat_cnt_and_scoresum[feat_name]['score'] += float(score)
                else:
                    feat_cnt_and_scoresum[feat_name] = {}
                    feat_cnt_and_scoresum[feat_name]['count'] = 1
                    feat_cnt_and_scoresum[feat_name]['score'] = float(score)

            CIS_by_fs[fs_name][term] = compute_CIS(feat_cnt_and_scoresum)


# CIS_sort = {
#     outer_key: dict(sorted(inner_dict.items(), key=lambda x: x[1], reverse=True))
#     for outer_key, inner_dict in CIS_by_fs.items()
# }


In [262]:
CIS_by_fs

{'ALL': {'long': {'use_m4': 3.4281694118690282,
   'dishwasherdisposalsinklight_m4': 3.4098930152233486,
   'kitchendenlights_m4': 3.400323440351554,
   'denoutdoorlights_m4': 3.396978826663259,
   'mudroomoutlets_m3': 3.381699925667087,
   'officelights_m4': 3.3671247267042146,
   'masteroutlets_m4': 3.344955648276186},
  'short': {'use_total': 4.422194830726624,
   'basementoutdooroutlets_m3': 4.367608061171238,
   'garageoutlets_m3': 4.357729931457535,
   'dishwasherdisposalsinklight_m4': 4.353724506729805,
   'microwave_m4': 4.3193495229007,
   'kitchenoutletseast_m4': 4.298193862578363,
   'denoutlets_m4': 4.244063603078769}},
 'STP': {'long': {'use_m4': 3.4281694118690282,
   'dishwasherdisposalsinklight_m4': 3.4098930152233486,
   'kitchendenlights_m4': 3.400323440351554,
   'denoutdoorlights_m4': 3.396978826663259,
   'mudroomoutlets_m3': 3.381699925667087,
   'officelights_m4': 3.3671247267042146,
   'masteroutlets_m4': 3.344955648276186},
  'short': {'use_total': 4.4221948307

In [263]:
CIS_by_fs = {}

for fs_name, all_info in extracted_features.items():
    for model_path, features in all_info.items():
        for term, feats in features.items():
            for feat_name, score in feats.items():
                if feat_name in feat_cnt_and_scoresum.keys():
                    feat_cnt_and_scoresum[feat_name]['count'] += 1
                    feat_cnt_and_scoresum[feat_name]['score'] += float(score)
                else:
                    feat_cnt_and_scoresum[feat_name] = {}
                    feat_cnt_and_scoresum[feat_name]['count'] = 1
                    feat_cnt_and_scoresum[feat_name]['score'] = float(score)

        CIS_by_fs[fs_name] = compute_CIS(feat_cnt_and_scoresum)


CIS_sort = {
    outer_key: dict(sorted(inner_dict.items(), key=lambda x: x[1], reverse=True))
    for outer_key, inner_dict in CIS_by_fs.items()
}


In [264]:
CIS_sort

{'ALL': {'use_total': 15.508687853588947,
  'dishwasherdisposalsinklight_m4': 13.73174965218605,
  'microwave_m4': 13.505357307822713,
  'masteroutlets_m4': 13.358704923241763,
  'denoutlets_m4': 12.801401083195346,
  'basementoutdooroutlets_m3': 11.902619167415512,
  'kitchendenlights_m4': 10.886455013381962,
  'garageoutlets_m3': 9.496774570289526,
  'rearbasementlights_m4': 8.437300183255543,
  'mudroomoutlets_m3': 8.046707128138952,
  'officelights_m4': 7.486902337706838,
  'denoutdoorlights_m4': 7.398103568450864,
  'kitchenoutletssouth_m4': 7.26440473314392,
  'use_m4': 7.2609079676286195,
  'kitchenoutletseast_m4': 7.008591475370474,
  'diningroomoutlets_m3': 6.919229505485008,
  'refrigerator_m4': 6.543059010702175,
  'masterbathoutlets_m3': 5.947977000247892,
  'masterbedbathlights_m4': 4.232108322865058,
  'sin_month': 0.4229009940313054,
  'windspeed': 0.3429438509674519,
  'hour': 0.32824327274848275},
 'STP': {'use_total': 21.49376689667579,
  'dishwasherdisposalsinklight_